## Script for statistics of binary relevance and classifier chains of benchmarks


In [3]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import utils
import os
from sklearn.metrics import jaccard_score, roc_auc_score

In [4]:
#setting filepaths of the results

pi_results_filepath = "../prediction_results/PI_results/PI_"

nnrti_results_filepath = "../prediction_results/NNRTI_results/NNRTI_"

nrti_results_filepath = "../prediction_results/NRTI_results/NRTI_"

ini_results_filepath = "../prediction_results/INI_results/INI_"



#creating lists with filepaths for later visualisation
filepath_list = [pi_results_filepath, nnrti_results_filepath, nrti_results_filepath, ini_results_filepath]




#setting filepaths the datasets

pi_results_filepath_ml = "../prediction_results/PI_results/benchmarkings/PI_"

nnrti_results_filepath_ml = "../prediction_results/NNRTI_results/benchmarkings/NNRTI_"

nrti_results_filepath_ml = "../prediction_results/NRTI_results/benchmarkings/NRTI_"

ini_results_filepath_ml = "../prediction_results/INI_results/benchmarkings/INI_"



#creating lists with filepaths for later visualisation
filepath_list_ml = [pi_results_filepath_ml, nnrti_results_filepath_ml, nrti_results_filepath_ml, ini_results_filepath_ml]



In [ ]:
files = ["Binary_Relevance_MOC_prediction.csv", "Binary_Relevance_homebrew_prediction.csv", "Classifier_Chain_homebrew_prediction.csv"]

sub_accs = {}
jacc_macro = {}


for path in filepath_list:

    acc_list = []

    for file in files:
        results = pd.read_csv(path + file)

        results.dropna(subset=results.columns[results.columns.str.startswith('True_')].tolist(), inplace=True)

        true_labels = results.filter(regex="True_*")
        pred_labels = results.filter(regex="Pred_*")

        sub_acc = roc_auc_score(true_labels, pred_labels, average="macro", multi_class="ovr")

        #print(jaccard_score(true_labels, pred_labels, average="samples"))
        #print(jaccard_score(true_labels, pred_labels, average="macro"))
        acc_list.append(sub_acc)

    sub_accs.update({path.split("/")[-1].strip("_"):acc_list})



print(sub_accs)



file = "_results/Binary_complete_labels_Multilabel_prediction.csv"


#models = ["", "RandomForest_", "XGBoost_", "CatBoost_"]

sub_accs_old = []

targets = ["PI", "NNRTI", "NRTI", "INI"]

for path, target in zip(filepath_list_ml, targets):

    df_old_ml_pred = pd.DataFrame()

    drugs = [file_name.strip("_results") for file_name in os.listdir(path) if "_results" in file_name]

    #acc_list = []


    for drug in drugs:
        results = pd.read_csv(path + drug + file)

        df_old_ml_pred["True_" + drug] = results["True"]

        #print(drug)

        df_old_ml_pred["Pred_" + drug] = [1.0 if proba >= 0.5 else 0.0 for proba in results["1"]]

    #print(df_old_ml_pred)

    true_labels = df_old_ml_pred.filter(regex="True_*")
    pred_labels = df_old_ml_pred.filter(regex="Pred_*")

    #print(true_labels)
    #print(pred_labels)

    sub_accs[target].append(jaccard_score(true_labels, pred_labels, average="macro"))

#print(sub_accs)

df_sub_accs = pd.DataFrame(sub_accs)



In [ ]:
models_w_TabPFN = ["BR w/o NaN", "BR w NaN", "CC", "BR w all label info"]

fig, axs = plt.subplots(2, 2, figsize=(12, 12))

axs = axs.flatten()

for i, drug in enumerate(df_sub_accs.columns.values.tolist()):

    print(axs[i])
    print(i)

    p = axs[i].bar(models_w_TabPFN, df_sub_accs[drug])

    axs[i].bar_label(p, label_type="edge")

    #axs[i].set_xticks(x)
    #axs[i].set_xticklabels(target_ml["Drug"])
    axs[i].set_ylabel('Jaccard Index macro averaged')
    axs[i].set_ylim(max(0, df_sub_accs[drug].min()* 0.995), min(1.0, df_sub_accs[drug].max()  * 1.015))
    axs[i].set_title('Comparison Binary relevance differing predictors: ' + drug)




#plt.savefig("../figures/BR_CC_comparison.png")


In [ ]:
#ml_models = ["BR_LR", "BR_XGB", "BR_forest", "CC_LR", "CC_xgb", "CC_forest", "Rakel_lr", "Rakel_xgb", "Rakel_forest"]

ml_models = ["BR_LR", "BR_XGB", "BR_forest", "CC_LR", "CC_xgb", "CC_forest"]

ending = ".csv"

sub_accs = {}

for path in filepath_list_ml:

    br_results = []
    cc_results = []
    #rakel_results = []

    for model in ml_models:
        results = pd.read_csv(path + model + ending)

        #results.dropna(subset=results.columns[results.columns.str.startswith('True_')].tolist(), inplace=True)

        true_labels = results.filter(regex="True_*")
        pred_labels = results.filter(regex="Pred_*")

        sub_acc = utils.subset_acc(pred_labels, true_labels)

        if model.startswith("BR"):
            br_results.append(sub_acc)
        elif model.startswith("CC"):
            cc_results.append(sub_acc)
        """
        else:
            rakel_results.append(sub_acc)
        """
    #sub_accs.update({path.split("/")[-1].strip("_"):[br_results, cc_results, rakel_results]})
    sub_accs.update({path.split("/")[-1].strip("_"):[br_results, cc_results]})


df_sub_accs = pd.DataFrame(sub_accs)

print(sub_accs)

In [ ]:
files = ["Binary_Relevance_MOC_prediction.csv", "Binary_Relevance_homebrew_prediction.csv", "Classifier_Chain_homebrew_prediction.csv"]

sub_accs_tabpfn = {}
jacc_macro = {}


for path in filepath_list:

    acc_list = []

    for file in files:
        results = pd.read_csv(path + file)

        results.dropna(subset=results.columns[results.columns.str.startswith('True_')].tolist(), inplace=True)

        true_labels = results.filter(regex="True_*")
        pred_labels = results.filter(regex="Pred_*")

        sub_acc = utils.subset_acc(true_labels, pred_labels, nan_mode="ignore")

        #print(jaccard_score(true_labels, pred_labels, average="samples"))
        #print(jaccard_score(true_labels, pred_labels, average="macro"))
        acc_list.append(sub_acc)

    sub_accs_tabpfn.update({path.split("/")[-1].strip("_"):acc_list})

print(sub_accs_tabpfn)

In [ ]:
test1 = list(np.array(["1", "2"]))
test2 = list(np.array(["3", "4"]))

print(test1 + test2)

In [ ]:
"""subset accuracy ignoring missing labels"""
"""
files = ["Binary_Relevance_MOC_prediction.csv", "Binary_Relevance_homebrew_prediction.csv", "Classifier_Chain_homebrew_prediction.csv"]

sub_accs = {}



for path in filepath_list_ml:

    acc_list = []

    for file in files:
        results = pd.read_csv(path + file)

        #results.dropna(subset=results.columns[results.columns.str.startswith('True_')].tolist(), inplace=True)

        true_labels = results.filter(regex="True_*")
        pred_labels = results.filter(regex="Pred_*")

        sub_acc = utils.subset_acc(pred_labels, true_labels, nan_mode="ignore")

        acc_list.append(sub_acc)

    sub_accs.update({path.split("/")[-1].strip("_"):acc_list})



print(sub_accs)



file = "_results/Binary_complete_labels_Multilabel_prediction.csv"


#models = ["", "RandomForest_", "XGBoost_", "CatBoost_"]

sub_accs_old = []

targets = ["PI", "NNRTI", "NRTI", "INI"]

for path, target in zip(filepath_list_ml, targets):

    df_old_ml_pred = pd.DataFrame()

    drugs = [file_name.strip("_results") for file_name in os.listdir(path) if "_results" in file_name]

    #acc_list = []


    for drug in drugs:
        results = pd.read_csv(path + drug + file)

        df_old_ml_pred["True_" + drug] = results["True"]

        #print(drug)

        df_old_ml_pred["Pred_" + drug] = [1.0 if proba >= 0.5 else 0.0 for proba in results["1"]]

    #print(df_old_ml_pred)

    true_labels = df_old_ml_pred.filter(regex="True_*")
    pred_labels = df_old_ml_pred.filter(regex="Pred_*")

    #print(true_labels)
    #print(pred_labels)

    sub_accs[target].append(utils.subset_acc(pred_labels, true_labels))

#print(sub_accs)

df_sub_accs = pd.DataFrame(sub_accs)
"""


models_w_TabPFN = ["BR w/o NaN", "BR w NaN", "CC"]

fig, axs = plt.subplots(2, 2, figsize=(12, 12))

axs = axs.flatten()

for i, drug in enumerate(df_sub_accs.columns.values.tolist()):

    benchmarks = np.array(sub_accs[drug]).flatten()
    results = list(benchmarks) + sub_accs_tabpfn[drug]
    labels = ml_models + models_w_TabPFN
    p = axs[i].bar(labels, results)

    axs[i].bar_label(p, label_type="edge")

    #axs[i].set_xticks(x)
    #axs[i].set_xticklabels(target_ml["Drug"])
    axs[i].set_ylabel('Subset accuracy')
    axs[i].set_ylim(max(0, min(results) * 0.995), min(1.0, max(results)  * 1.015))
    axs[i].set_title('Comparison Binary relevance differing predictors: ' + drug)




#plt.savefig("../figures/BR_CC_comparison_woNaN.png")



In [ ]:
ml_models = ["BR_LR", "BR_XGB", "BR_forest", "CC_LR", "CC_xgb", "CC_forest"]

ending = ".csv"

sub_accs = {}

for path in filepath_list_ml:

    br_results = []
    cc_results = []
    #rakel_results = []

    for model in ml_models:
        results = pd.read_csv(path + model + ending)

        #results.dropna(subset=results.columns[results.columns.str.startswith('True_')].tolist(), inplace=True)

        true_labels = results.filter(regex="True_*")
        pred_labels = results.filter(regex="Pred_*")

        sub_acc = utils.exam_acc(pred_labels, true_labels)

        if model.startswith("BR"):
            br_results.append(sub_acc)
        elif model.startswith("CC"):
            cc_results.append(sub_acc)

        #else:
        #    rakel_results.append(sub_acc)

    #sub_accs.update({path.split("/")[-1].strip("_"):[br_results, cc_results, rakel_results]})
    sub_accs.update({path.split("/")[-1].strip("_"):[br_results, cc_results]})

files = ["Binary_Relevance_MOC_prediction.csv", "Binary_Relevance_homebrew_prediction.csv", "Classifier_Chain_homebrew_prediction.csv"]

sub_accs_tabpfn = {}
jacc_macro = {}


for path in filepath_list:

    acc_list = []

    for file in files:
        results = pd.read_csv(path + file)

        results.dropna(subset=results.columns[results.columns.str.startswith('True_')].tolist(), inplace=True)

        true_labels = results.filter(regex="True_*")
        pred_labels = results.filter(regex="Pred_*")

        sub_acc = utils.exam_acc(true_labels, pred_labels)

        #print(jaccard_score(true_labels, pred_labels, average="samples"))
        #print(jaccard_score(true_labels, pred_labels, average="macro"))
        acc_list.append(sub_acc)

    sub_accs_tabpfn.update({path.split("/")[-1].strip("_"):acc_list})



df_sub_accs = pd.DataFrame(sub_accs)



models_w_TabPFN = ["BR w/o NaN", "BR w NaN", "CC"]

fig, axs = plt.subplots(2, 2, figsize=(18, 12))

axs = axs.flatten()

for i, drug in enumerate(df_sub_accs.columns.values.tolist()):

    benchmarks = np.array(sub_accs[drug]).flatten()
    results = list(benchmarks) + sub_accs_tabpfn[drug]
    labels = ml_models + models_w_TabPFN
    p = axs[i].bar(labels, results)

    axs[i].bar_label(p, label_type="edge")

    #axs[i].set_xticks(x)
    #axs[i].set_xticklabels(target_ml["Drug"])
    axs[i].tick_params(rotation=45)
    axs[i].set_ylabel('Subset accuracy')
    axs[i].set_ylim(max(0, min(results) * 0.995), min(1.0, max(results)  * 1.015))
    axs[i].set_title('Comparison Binary relevance differing predictors: ' + drug)



#plt.savefig("../figures/BR_CC_comparison_woNaN.png")



In [ ]:
files = ["Binary_Relevance_5_fold_MOC_prediction_new_save.csv", "Binary_Relevance_5_fold_homebrew_prediction_new_save.csv", "Classifier_Chain_5_fold_homebrew_prediction_new_save.csv"]


sub_accs_fold_mean = {}
sub_accs_fold_std = {}

for path in filepath_list:
    acc_list_mean = []
    acc_list_std = []

    for file in files:
        results = pd.read_csv(path + file)

        #results.dropna(subset=results.columns[results.columns.str.startswith('True_')].tolist(), inplace=True)

        subs_accs_groups = results.groupby(by="kFolds").apply(lambda x: utils.subset_acc(x.filter(regex="Pred_*"), x.filter(regex="True_*"), nan_mode="ignore"), include_groups=False)


        acc_list_mean.append(subs_accs_groups.mean())
        acc_list_std.append(subs_accs_groups.std())

    sub_accs_fold_mean.update({path.split("/")[-1].strip("_"):acc_list_mean})
    sub_accs_fold_std.update({path.split("/")[-1].strip("_"):acc_list_std})


#print(sub_accs)

#df_sub_accs_groups = pd.DataFrame(subs_accs_groups)

df_sub_accs_groups_mean = pd.DataFrame(sub_accs_fold_mean)
df_sub_accs_groups_std = pd.DataFrame(sub_accs_fold_std)

#plt.bar(df_sub_accs)

models_w_TabPFN = ["BR w/o NaN", "BR w NaN", "CC"]

fig, axs = plt.subplots(2, 2, figsize=(12, 12))

axs = axs.flatten()

for i, drug in enumerate(df_sub_accs_groups_mean.columns.values.tolist()):

    #print(axs[i])
    #print(i)

    p = axs[i].bar(models_w_TabPFN, df_sub_accs_groups_mean[drug])
    axs[i].errorbar(models_w_TabPFN, df_sub_accs_groups_mean[drug], yerr=df_sub_accs_groups_std[drug],capsize=3, fmt="none", ecolor="black")

    axs[i].bar_label(p, label_type="edge")

    #axs[i].set_xticks(x)
    #axs[i].set_xticklabels(target_ml["Drug"])
    axs[i].set_ylabel('Subset accuracy')
    axs[i].set_ylim(max(0, (df_sub_accs_groups_mean[drug].min() - df_sub_accs_groups_std[drug].max()) * 0.995), min(1.0 + df_sub_accs_groups_std[drug].max(), (df_sub_accs_groups_mean[drug].max() + df_sub_accs_groups_std[drug].max())  * 1.015))
    axs[i].set_title('Comparison Binary relevance differing predictors: ' + drug)


#plt.savefig("../figures/BR_CC_kfold_model_comparison_CCownCV.png")

    #plt.savefig("../figures/" + filepath_ml.split("/")[-1].split("_")[0] + "_Multilabel_Comparison.png")


In [5]:
ml_models = ["BR_LR", "BR_XGB", "BR_forest", "CC_LR", "CC_xgb", "CC_forest"]

ending = "_5_fold.csv"


for path in filepath_list_ml:

    acc_list_mean = []
    acc_list_std = []
    #rakel_results = []

    for model in ml_models:

        results = pd.read_csv(path + model + ending)

        #results.dropna(subset=results.columns[results.columns.str.startswith('True_')].tolist(), inplace=True)

        subs_accs_groups = results.groupby(by="kFolds").apply(lambda x: utils.subset_acc(x.filter(regex="Pred_*"), x.filter(regex="True_*"), nan_mode="ignore"), include_groups=False)


        acc_list_mean.append(subs_accs_groups.mean())
        acc_list_std.append(subs_accs_groups.std())

        #else:
        #    rakel_results.append(sub_acc)

    #sub_accs.update({path.split("/")[-1].strip("_"):[br_results, cc_results, rakel_results]})
    sub_accs_fold_mean.update({path.split("/")[-1].strip("_"):acc_list_mean})
    sub_accs_fold_std.update({path.split("/")[-1].strip("_"):acc_list_std})



files = ["Binary_Relevance_5_fold_MOC_prediction_new_save.csv", "Binary_Relevance_5_fold_homebrew_prediction_new_save.csv", "Classifier_Chain_5_fold_homebrew_prediction_new_save.csv"]


sub_accs_fold_mean = {}
sub_accs_fold_std = {}

for path in filepath_list:
    acc_list_mean = []
    acc_list_std = []

    for file in files:
        results = pd.read_csv(path + file)

        #results.dropna(subset=results.columns[results.columns.str.startswith('True_')].tolist(), inplace=True)

        subs_accs_groups = results.groupby(by="kFolds").apply(lambda x: utils.subset_acc(x.filter(regex="Pred_*"), x.filter(regex="True_*"), nan_mode="ignore"), include_groups=False)


        acc_list_mean.append(subs_accs_groups.mean())
        acc_list_std.append(subs_accs_groups.std())

    sub_accs_fold_mean[path.split("/")[-1].strip("_")].extend(acc_list_mean)
    sub_accs_fold_std[path.split("/")[-1].strip("_")].extend(acc_list_std)




models_w_TabPFN = ["BR w/o NaN", "BR w NaN", "CC"]

fig, axs = plt.subplots(2, 2, figsize=(18, 12))

axs = axs.flatten()

for i, drug in enumerate(df_sub_accs.columns.values.tolist()):


    labels = ml_models + models_w_TabPFN
    p = axs[i].bar(labels, df_sub_accs_groups_mean[drug])
    axs[i].errorbar(labels, df_sub_accs_groups_mean[drug], yerr=df_sub_accs_groups_std[drug],capsize=3, fmt="none", ecolor="black")


    axs[i].bar_label(p, label_type="edge")

    #axs[i].set_xticks(x)
    #axs[i].set_xticklabels(target_ml["Drug"])
    axs[i].tick_params(rotation=45)
    axs[i].set_ylabel('Subset accuracy')
    axs[i].set_ylim(max(0, (df_sub_accs_groups_mean[drug].min() - df_sub_accs_groups_std[drug].max()) * 0.995), min(1.0 + df_sub_accs_groups_std[drug].max(), (df_sub_accs_groups_mean[drug].max() + df_sub_accs_groups_std[drug].max())  * 1.015))
    axs[i].set_title('Comparison Binary relevance differing predictors: ' + drug)



#plt.savefig("../figures/BR_CC_comparison_woNaN.png")



KeyError: 'kFolds'

In [ ]:
my_dict = {
    'fruits': ['apple', 'banana', 'cherry']
}

# Add multiple fruits from another list
new_fruits = ['grape', 'mango']
my_dict['fruits'].extend(new_fruits)

print(my_dict)
# Output: {'fruits': ['apple', 'banana', 'cherry', 'grape', 'mango']}